# Sesión 2 · Correcto: definir «correcto» antes de construir

**Objetivo.** Escribir una especificación del análisis, agregarla al agente, y construir chequeos
automáticos que digan si cada corrida la cumple. **Tiempo:** 80 minutos.

In [ ]:
# --- Configuración: correr una vez al empezar ---
import os, sys, json
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
import pandas as pd
import matplotlib.pyplot as plt
from agente.llm import desde_entorno
from agente.harness import ejecutar_agente, correr_varias, leer_traza, con_especificacion, SISTEMA_BASE
from agente.evaluacion import tarjeta, huerfanos, guardar, cargar, cargar_respuesta, diagnosticos_ejecutados

# os.environ["LLM_PROVEEDOR"] = "ollama"   # o "anthropic" / "openai"; si no lo configurás, se usa el simulador
# os.environ["LLM_MODELO"] = "qwen3.5:0.8b"   # el modelo que descargaste (ver INSTALACION.md)
llm = desde_entorno()
df = pd.read_csv(RAIZ / "datos" / "programa_x.csv")
TRAZAS = RAIZ / "trazas" / "s2"
RES = RAIZ / "resultados"; RES.mkdir(exist_ok=True)
VERDAD, TOLERANCIA = cargar_respuesta(RAIZ)
PREGUNTA = "¿Cuánto aumentó el programa el ingreso posterior de quienes participaron?"

## 1. La especificación (25 min)

Completá `especificaciones/especificacion.md` en equipo. Funciona como un pre-registro: decide qué
cuenta como correcto **antes** de ver resultados. Cada sección que dejes vacía la decide el agente por vos.

Cuando termines, corré la celda: la especificación se agrega al mensaje de sistema.

In [ ]:
ESPEC = (RAIZ / "especificaciones" / "especificacion.md").read_text(encoding="utf-8")
SISTEMA = con_especificacion(ESPEC)
print(SISTEMA[-1500:])

## 2. Chequeos automáticos (20 min)

Un chequeo de código mira lo que el agente **ejecutó**, no lo que dice que hizo. Estos son los del kit:

In [ ]:
from agente.evaluacion import PATRONES
PATRONES

Son búsquedas de texto en el código: baratas, rápidas y frágiles. **Ejercicio:** escribí un chequeo
propio. Ideas: ¿la respuesta declara el estimando? ¿El intervalo contiene a la estimación?
¿Usó todas las covariables de la especificación?

In [ ]:
def mi_chequeo(fila) -> bool:
    """True si la corrida pasa. `fila` es una fila de lo que devuelve correr_varias."""
    # TODO: reemplazá este ejemplo por tu chequeo
    return pd.notna(fila["ic_inferior"]) and fila["ic_inferior"] <= fila["estimacion"] <= fila["ic_superior"]

## 3. Cinco corridas con especificación (15 min)

In [ ]:
res = correr_varias(5, PREGUNTA, df, llm, sistema=SISTEMA, carpeta_trazas=TRAZAS, etiqueta="s2")
guardar(res, RES / "s2.json")
res["mi_chequeo"] = res.apply(mi_chequeo, axis=1)
res[["metodo", "estimacion", "ic_inferior", "ic_superior", "mi_chequeo"]]

## 4. ¿Mejoró? Línea base contra especificación (5 min)

In [ ]:
base = cargar(RES / "s1.json")
tarjeta(base, VERDAD, TOLERANCIA).merge(tarjeta(res, VERDAD, TOLERANCIA),
                                        on=["pilar", "métrica"], suffixes=(" · s1", " · s2"))

## 5. Dos preguntas incómodas (15 min)

**¿El modelo reconoce los datos?** Si un modelo leyó sobre este estudio, puede responder de memoria
en lugar de analizar. Por eso los datos vienen disfrazados. Probalo:

In [ ]:
if llm.nombre != "simulador":
    resumen = df.describe().T.round(0).to_string()
    print(llm.completar("Respondé en dos frases.", [{"role": "user", "content":
        f"Resumen de un conjunto de datos:\n{resumen}\n¿Reconocés de qué estudio viene? Si sí, ¿cuál es su resultado conocido?"}]))
else:
    print("Con el simulador no aplica: probalo con un modelo real.")

**¿Quién juzga?** Un modelo como juez es flexible pero puede creerle al informe. Comparalo con
tus chequeos de código: ¿en qué casos dice que se corrió un diagnóstico que el código no muestra?

In [ ]:
JUEZ = ('Sos un evaluador estricto. Te paso una especificación y un informe. '
        'Respondé solo con JSON: {"cumple": true/false, "faltantes": [...]}')
if llm.nombre != "simulador":
    for _, fila in res.iterrows():
        informe = fila[["estimacion", "ic_inferior", "ic_superior", "metodo", "diagnosticos", "afirmaciones"]].to_json()
        veredicto = llm.completar(JUEZ, [{"role": "user", "content": f"ESPECIFICACIÓN:\n{ESPEC}\n\nINFORME:\n{informe}"}])
        print(fila.corrida, "| juez:", veredicto.strip()[:200], "| código:", sorted(diagnosticos_ejecutados(fila.traza)))

## Para la próxima

- **Leer:** Silberzahn et al. (2018), «Many Analysts, One Data Set»; Bertran, Fogliato & Wu (2026), «Many AI Analysts, One Dataset».
- **Traer:** tu especificación. La vamos a someter a veinte corridas.